# Introduction
This notebook is a clone of the run_quantize.py file for mimicking and debugging experiments without using seml and slurm

# 1. Initial Setup

In [1]:
print("Hello World")
!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'

Hello World
Free GPU Memory (GB): 39.3936


In [2]:
# Setting up environment
print("\n################################")
print("Setting up environment...")
print("################################\n")

import os
#os.chdir('..')
print("Current Working Directory ", os.getcwd())
import sys
sys.path.append("../") # Add directory containing src/data to path

import importlib
import src  # Assuming src is the package name

# Reload the src module after making changes
importlib.reload(src)

%load_ext autoreload
%autoreload 2

import seml
import re
import shutil

os.environ["TOKENIZERS_PARALLELISM"] = "false"  # Disables parallelism to remove transformers warning

print("\n################################")
print("Setting up cache paths...")
print("################################\n")

os.environ["MKL_SERVICE_FORCE_INTEL"] = "1"
CACHE_PATH = "/nfs/students/daro/.cache/huggingface"
HUB_PATH = "/nfs/students/daro/.cache/huggingface/hub/"

if not os.path.exists(HUB_PATH):
    os.makedirs(HUB_PATH)
    print(f"Creating huggingface hub path at {HUB_PATH}")
    
print(f"Setting cache path to {CACHE_PATH}")
os.environ["TORCH_HOME"] = CACHE_PATH
os.environ["HF_HOME"] = CACHE_PATH

import torch
torch.hub.set_dir(CACHE_PATH)
with torch.no_grad():
    torch.cuda.empty_cache()
    
import logging
logger = logging.getLogger("quant_logger")
    
!cat /proc/meminfo | awk '/MemTotal/ {total=$2} /MemFree/ {free=$2} /MemAvailable/ {available=$2} END {printf "MemTotal: %.2f GB\nMemFree: %.2f GB\nMemAvailable: %.2f GB\n", total/1024/1024, free/1024/1024, available/1024/1024}'
!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'

print("\n################################")
print("Setting up cuda devices...")
print("################################\n")

if torch.cuda.is_available():
    print("CUDA device is available!")
    # Get the number of available CUDA devices
    num_cuda_devices = torch.cuda.device_count()
    print(f"Number of CUDA devices: {num_cuda_devices}")
    
    # Loop through available devices and get name
    for device_id in range(num_cuda_devices):
        device = torch.device(f"cuda:{device_id}")
        name = torch.cuda.get_device_name(device)
        print(f"  - CUDA Device {device_id+1}: {name}")
else:
    print("CUDA device is not available.")
    
print("\n################################")
print("Authentication with Hugging Face...")
print("################################\n")

import os
from dotenv import load_dotenv
from huggingface_hub import login

load_dotenv()
huggingface_token = os.getenv('HUGGINGFACE_TOKEN')

if huggingface_token is None:
    raise ValueError("Please set the HUGGINGFACE_TOKEN environment variable.")
else:
    print("Hugging Face token loaded successfully.")

login(token=huggingface_token, add_to_git_credential=True)
print("Successfully authenticated with the Hugging Face API.")

print("\n################################")
print("Setting up GPU memory usage list...")
print("################################\n")
# Global list to store GPU memory usage
from src.evaluations.evaluate_memory import record_gpu_memory
gpu_memory_usage = {}
record_gpu_memory(gpu_memory_usage=gpu_memory_usage, context="Warm up notebook")


################################
Setting up environment...
################################

Current Working Directory  /nfs/homedirs/daro/git/quantization-reliability
Initializing src package
Initializing src package

################################
Setting up cache paths...
################################

Setting cache path to /nfs/students/daro/.cache/huggingface
MemTotal: 1007.72 GB
MemFree: 375.31 GB
MemAvailable: 950.98 GB
Free GPU Memory (GB): 39.3936

################################
Setting up cuda devices...
################################

CUDA device is available!
Number of CUDA devices: 1
  - CUDA Device 1: NVIDIA A100-PCIE-40GB

################################
Authentication with Hugging Face...
################################

Hugging Face token loaded successfully.
Token is valid (permission: write).
Your token has been saved in your configured git credential helpers (store).
Your token has been saved to /nfs/students/daro/.cache/huggingface/token
Login successfu

# 2. Debug Taxonomy for Llama and AWQ

In [3]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
import pandas as pd
from src.reliability.response_generator import ResponseGenerator
from src.data.constants import DATA_DIR, DATA_FILES

def load_taxonomy_qa_pairs(data_dir, dataset_name="P101", max_samples=5, typo_type="word_taxonomy_neg"):
    """Load QA pairs with taxonomy perturbations"""
    import os
    import json
    
    file_path = os.path.join(data_dir, f"{dataset_name}-subclass.json")
    qa_pairs = []
    
    if os.path.exists(file_path):
        with open(file_path, 'r', encoding='utf8') as f:
            lines = [json.loads(line) for line in f.readlines()]
            
        if len(lines) < 2:
            raise ValueError(f"Insufficient data in file: {file_path}")
            
        relations = lines[0]['relations'][:1]  # Take only first relation for simplicity
        entries = lines[1:max_samples+1]
        
        for entry in entries:
            subject = entry['subject']
            answer = entry['object']
            taxonomy = entry.get('taxonomy', [])
            
            for relation in relations:
                question = relation.replace("[X]", subject)
                
                # Apply taxonomy perturbation logic here if needed
                # For now, we'll just return the base pairs
                qa_pairs.append((question, answer))
                
    return qa_pairs

def run_model_comparison(base_model_name, quantized_model_name, dataset_name="P101", max_samples=5):
    """Run comparison between base and quantized models"""
    
    # Load QA pairs for both perturbation types
    neg_pairs = load_taxonomy_qa_pairs(DATA_DIR, dataset_name, max_samples, "word_taxonomy_neg")
    pos_pairs = load_taxonomy_qa_pairs(DATA_DIR, dataset_name, max_samples, "word_taxonomy_pos")
    
    # Initialize response generators for both models
    base_generator = ResponseGenerator(base_model_name, cache_dir=None)
    quant_generator = ResponseGenerator(quantized_model_name, cache_dir=None)
    
    results = []
    
    # Process both negative and positive taxonomy pairs
    for pairs, taxonomy_type in [(neg_pairs, "negative"), (pos_pairs, "positive")]:
        queries, true_answers = zip(*pairs)
        
        # Generate responses for both models
        base_responses = base_generator.generate_responses(
            queries=queries,
            strategy="Direct Completion",
            dataset_name=dataset_name,
            true_answers=true_answers,
            max_new_tokens=25,
            temperature=0.1,
            use_beam_search=False,
            n_repeats=1
        )
        
        quant_responses = quant_generator.generate_responses(
            queries=queries,
            strategy="Direct Completion",
            dataset_name=dataset_name,
            true_answers=true_answers,
            max_new_tokens=25,
            temperature=0.1,
            use_beam_search=False,
            n_repeats=1
        )
        
        # Collect results
        for i, (query, true_answer) in enumerate(pairs):
            base_result = base_responses[i][0]  # Take first response since n_repeats=1
            quant_result = quant_responses[i][0]
            
            results.append({
                "Taxonomy": taxonomy_type,
                "Query": query,
                "True Answer": true_answer,
                "Base Model Response": base_result["output_text"],
                "Base Model Cleaned": base_result["cleaned"],
                "Base Model Probability": base_result["beam_prob"],
                "Base Model Adjusted Prob": base_result["beam_prob_adj"],
                "Base Model Entropy": base_result["entropy"],
                "Base Model Correct": base_result["is_correct"],
                "Quantized Model Response": quant_result["output_text"],
                "Quantized Model Cleaned": quant_result["cleaned"],
                "Quantized Model Probability": quant_result["beam_prob"],
                "Quantized Model Adjusted Prob": quant_result["beam_prob_adj"],
                "Quantized Model Entropy": quant_result["entropy"],
                "Quantized Model Correct": quant_result["is_correct"]
            })
    
    # Create DataFrame and save to Excel
    df = pd.DataFrame(results)
    excel_path = "taxonomy_comparison_results.xlsx"
    df.to_excel(excel_path, index=False)
    print(f"Results saved to {excel_path}")
    
    return df

if __name__ == "__main__":
    # Example usage
    base_model = "Llama-3-8B"
    quantized_model = "Llama-3-8B-AWQ-4bit-local"
    
    results_df = run_model_comparison(
        base_model_name=base_model,
        quantized_model_name=quantized_model,
        dataset_name="P101",
        max_samples=100
    )
    
    # Print summary statistics
    print("\nSummary Statistics:")
    print("Base Model Accuracy:", results_df["Base Model Correct"].mean())
    print("Quantized Model Accuracy:", results_df["Quantized Model Correct"].mean())
    
    # Compare performance by taxonomy type
    print("\nPerformance by Taxonomy Type:")
    taxonomy_stats = results_df.groupby("Taxonomy").agg({
        "Base Model Correct": "mean",
        "Quantized Model Correct": "mean",
        "Base Model Entropy": "mean",
        "Quantized Model Entropy": "mean"
    })
    print(taxonomy_stats)

2024-10-29 23:50:11.594947: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-10-29 23:50:11.613500: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-10-29 23:50:11.618995: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-10-29 23:50:11.633272: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-10-29 23:50:13.263094: W tensorflow/compiler/tf2

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
Tokenizer model max length reduced from 1000000000000000019884624838656 to 2048 to fit in memory
Unexpected error occurred while loading model: 'NoneType' object has no attribute 'NAME'


AttributeError: 'NoneType' object has no attribute 'NAME'

### Calculate number of correct rows

In [8]:
import os
import pandas as pd
import glob
import re

def extract_config(filename):
    pattern = r"(.+?)_(.+?)_(.+?)_typo(\d+)_(.+?)_tok(\d+)_temp([\d.]+)_(.+?)_rep(\d+)_beams(\d+)_maxent(.+?)_rows(\d+)"
    match = re.match(pattern, filename)
    if match:
        return {
            'model_name': match.group(1),
            'dataset_name': match.group(2),
            'typo_type': match.group(3),
            'typo_intensity': match.group(4),
            'beam_search_str': match.group(5),
            'max_new_tokens': match.group(6),
            'temperature': match.group(7),
            'strategy_str': match.group(8),
            'n_repeats': match.group(9),
            'n_beams': match.group(10),
            'max_entries': match.group(11),
            'num_excel_rows': match.group(12)
        }
    return None

def count_correct_predictions(exp_id, model_name, typo_type='char_insertion'):
    results_path = "/nfs/homedirs/daro/git/quantization-reliability/results"
    exp_dir = os.path.join(results_path, "reliability_eval", f"reliability_eval_{exp_id}")
    
    files = glob.glob(os.path.join(exp_dir, f"{model_name}*raw_table*.xlsx"))
    
    results = {}
    
    for file in files:
        config = extract_config(os.path.basename(file))
        if config and config['typo_type'] == typo_type:
            df = pd.read_excel(file)
            dataset = config['dataset_name']
            if dataset not in results:
                results[dataset] = {'correct': 0, 'total': 0}
            results[dataset]['correct'] += df['Is Correct'].sum()
            results[dataset]['total'] += len(df)
    
    return results

def main(exp_id, model1, model2):
    print(f"Analyzing experiment: {exp_id}")
    print(f"Comparing models: {model1} vs {model2}")
    print(f"Filtering for typo_type: char_insertion")
    
    results1 = count_correct_predictions(exp_id, model1)
    results2 = count_correct_predictions(exp_id, model2)
    
    all_datasets = set(list(results1.keys()) + list(results2.keys()))
    
    for dataset in all_datasets:
        print(f"\nResults for dataset: {dataset}")
        
        correct1 = results1.get(dataset, {'correct': 0, 'total': 0})['correct']
        total1 = results1.get(dataset, {'correct': 0, 'total': 0})['total']
        correct2 = results2.get(dataset, {'correct': 0, 'total': 0})['correct']
        total2 = results2.get(dataset, {'correct': 0, 'total': 0})['total']
        
        accuracy1 = correct1 / total1 if total1 > 0 else 0
        accuracy2 = correct2 / total2 if total2 > 0 else 0
        
        print(f"{model1}:")
        print(f"  Total correct predictions: {correct1}")
        print(f"  Total rows: {total1}")
        print(f"  Accuracy: {accuracy1:.4f}")
        
        print(f"\n{model2}:")
        print(f"  Total correct predictions: {correct2}")
        print(f"  Total rows: {total2}")
        print(f"  Accuracy: {accuracy2:.4f}")
        
        print(f"\nAccuracy difference ({model2} - {model1}): {accuracy2 - accuracy1:.4f}")
    
    # Calculate and print overall results
    total_correct1 = sum(results['correct'] for results in results1.values())
    total_rows1 = sum(results['total'] for results in results1.values())
    total_correct2 = sum(results['correct'] for results in results2.values())
    total_rows2 = sum(results['total'] for results in results2.values())
    
    overall_accuracy1 = total_correct1 / total_rows1 if total_rows1 > 0 else 0
    overall_accuracy2 = total_correct2 / total_rows2 if total_rows2 > 0 else 0
    
    print("\nOverall Results:")
    print(f"{model1} overall accuracy: {overall_accuracy1:.4f}")
    print(f"{model2} overall accuracy: {overall_accuracy2:.4f}")
    print(f"Overall accuracy difference ({model2} - {model1}): {overall_accuracy2 - overall_accuracy1:.4f}")

if __name__ == "__main__":
    exp_id = "pert-awq-bnb-hqq-10-07"
    model1 = "Llama-3-8B"
    model2 = "Llama-3-8B-HQQ-mixed-local"
    main(exp_id, model1, model2)

Analyzing experiment: pert-awq-bnb-hqq-10-07
Comparing models: Llama-3-8B vs Llama-3-8B-HQQ-mixed-local
Filtering for typo_type: char_insertion

Results for dataset: P30
Llama-3-8B:
  Total correct predictions: 650
  Total rows: 1200
  Accuracy: 0.5417

Llama-3-8B-HQQ-mixed-local:
  Total correct predictions: 146
  Total rows: 300
  Accuracy: 0.4867

Accuracy difference (Llama-3-8B-HQQ-mixed-local - Llama-3-8B): -0.0550

Results for dataset: P740
Llama-3-8B:
  Total correct predictions: 455
  Total rows: 1200
  Accuracy: 0.3792

Llama-3-8B-HQQ-mixed-local:
  Total correct predictions: 88
  Total rows: 300
  Accuracy: 0.2933

Accuracy difference (Llama-3-8B-HQQ-mixed-local - Llama-3-8B): -0.0858

Results for dataset: P364
Llama-3-8B:
  Total correct predictions: 688
  Total rows: 1200
  Accuracy: 0.5733

Llama-3-8B-HQQ-mixed-local:
  Total correct predictions: 155
  Total rows: 300
  Accuracy: 0.5167

Accuracy difference (Llama-3-8B-HQQ-mixed-local - Llama-3-8B): -0.0567

Results for da

### calculate accuracies for each dataset

In [10]:
import os
import pandas as pd
import glob
import re

def extract_config(filename):
    pattern = r"(.+?)_(.+?)_(.+?)_typo(\d+)_(.+?)_tok(\d+)_temp([\d.]+)_(.+?)_rep(\d+)_beams(\d+)_maxent(.+?)_rows(\d+)"
    match = re.match(pattern, filename)
    if match:
        return {
            'model_name': match.group(1),
            'dataset_name': match.group(2),
            'typo_type': match.group(3),
            'typo_intensity': match.group(4),
            'beam_search_str': match.group(5),
            'max_new_tokens': match.group(6),
            'temperature': match.group(7),
            'strategy_str': match.group(8),
            'n_repeats': match.group(9),
            'n_beams': match.group(10),
            'max_entries': match.group(11),
            'num_excel_rows': match.group(12)
        }
    return None

def count_correct_predictions(exp_id, model_name, typo_type='char_insertion'):
    results_path = "/nfs/homedirs/daro/git/quantization-reliability/results"
    exp_dir = os.path.join(results_path, "reliability_eval", f"reliability_eval_{exp_id}")
    
    files = glob.glob(os.path.join(exp_dir, f"{model_name}*raw_table*.xlsx"))
    
    results = {}
    
    for file in files:
        config = extract_config(os.path.basename(file))
        if config and config['typo_type'] == typo_type:
            df = pd.read_excel(file)
            dataset = config['dataset_name']
            if dataset not in results:
                results[dataset] = {'correct': 0, 'total': 0}
            results[dataset]['correct'] += df['Is Correct'].sum()
            results[dataset]['total'] += len(df)
    
    return results

def main(exp_id, model1, model2):
    print(f"Analyzing experiment: {exp_id}")
    print(f"Comparing models: {model1} vs {model2}")
    print(f"Filtering for typo_type: char_insertion")
    
    results1 = count_correct_predictions(exp_id, model1)
    results2 = count_correct_predictions(exp_id, model2)
    
    all_datasets = set(list(results1.keys()) + list(results2.keys()))
    
    for dataset in all_datasets:
        print(f"\nResults for dataset: {dataset}")
        
        correct1 = results1.get(dataset, {'correct': 0, 'total': 0})['correct']
        total1 = results1.get(dataset, {'correct': 0, 'total': 0})['total']
        correct2 = results2.get(dataset, {'correct': 0, 'total': 0})['correct']
        total2 = results2.get(dataset, {'correct': 0, 'total': 0})['total']
        
        accuracy1 = correct1 / total1 if total1 > 0 else 0
        accuracy2 = correct2 / total2 if total2 > 0 else 0
        
        print(f"{model1}:")
        print(f"  Total correct predictions: {correct1}")
        print(f"  Total rows: {total1}")
        print(f"  Accuracy: {accuracy1:.4f}")
        
        print(f"\n{model2}:")
        print(f"  Total correct predictions: {correct2}")
        print(f"  Total rows: {total2}")
        print(f"  Accuracy: {accuracy2:.4f}")
        
        print(f"\nAccuracy difference ({model2} - {model1}): {accuracy2 - accuracy1:.4f}")
    
    # Calculate and print overall results
    total_correct1 = sum(results['correct'] for results in results1.values())
    total_rows1 = sum(results['total'] for results in results1.values())
    total_correct2 = sum(results['correct'] for results in results2.values())
    total_rows2 = sum(results['total'] for results in results2.values())
    
    overall_accuracy1 = total_correct1 / total_rows1 if total_rows1 > 0 else 0
    overall_accuracy2 = total_correct2 / total_rows2 if total_rows2 > 0 else 0
    
    print("\nOverall Results:")
    print(f"{model1} overall accuracy: {overall_accuracy1:.4f}")
    print(f"{model2} overall accuracy: {overall_accuracy2:.4f}")
    print(f"Overall accuracy difference ({model2} - {model1}): {overall_accuracy2 - overall_accuracy1:.4f}")

if __name__ == "__main__":
    exp_id = "pert-awq-bnb-hqq-10-07"
    model1 = "Llama-3-8B"
    model2 = "Llama-3-8B-HQQ-mixed-local"
    main(exp_id, model1, model2)

Analyzing experiment: pert-awq-bnb-hqq-10-07
Comparing models: Llama-3-8B vs Llama-3-8B-HQQ-mixed-local
Filtering for typo_type: char_insertion

Results for dataset: P30
Llama-3-8B:
  Total correct predictions: 650
  Total rows: 1200
  Accuracy: 0.5417

Llama-3-8B-HQQ-mixed-local:
  Total correct predictions: 146
  Total rows: 300
  Accuracy: 0.4867

Accuracy difference (Llama-3-8B-HQQ-mixed-local - Llama-3-8B): -0.0550

Results for dataset: P740
Llama-3-8B:
  Total correct predictions: 455
  Total rows: 1200
  Accuracy: 0.3792

Llama-3-8B-HQQ-mixed-local:
  Total correct predictions: 88
  Total rows: 300
  Accuracy: 0.2933

Accuracy difference (Llama-3-8B-HQQ-mixed-local - Llama-3-8B): -0.0858

Results for dataset: P364
Llama-3-8B:
  Total correct predictions: 688
  Total rows: 1200
  Accuracy: 0.5733

Llama-3-8B-HQQ-mixed-local:
  Total correct predictions: 155
  Total rows: 300
  Accuracy: 0.5167

Accuracy difference (Llama-3-8B-HQQ-mixed-local - Llama-3-8B): -0.0567

Results for da

### Calculate excel number of rows

In [14]:
import os
import pandas as pd
import glob
import re
from collections import defaultdict

def extract_config(filename):
    pattern = r"(.+?)_(.+?)_(.+?)_typo(\d+)_(.+?)_tok(\d+)_temp([\d.]+)_(.+?)_rep(\d+)_beams(\d+)_maxent(.+?)_rows(\d+)"
    match = re.match(pattern, filename)
    if match:
        return {
            'model_name': match.group(1),
            'dataset_name': match.group(2),
            'typo_type': match.group(3),
            'typo_intensity': match.group(4),
            'beam_search_str': match.group(5),
            'max_new_tokens': match.group(6),
            'temperature': match.group(7),
            'strategy_str': match.group(8),
            'n_repeats': match.group(9),
            'n_beams': match.group(10),
            'max_entries': match.group(11),
            'num_excel_rows': match.group(12)
        }
    return None

def analyze_excel_files(exp_id):
    results_path = "/nfs/homedirs/daro/git/quantization-reliability/results"
    exp_dir = os.path.join(results_path, "reliability_eval", f"reliability_eval_{exp_id}")
    
    files = glob.glob(os.path.join(exp_dir, "*raw_table*.xlsx"))
    
    row_counts = defaultdict(list)
    
    for file in files:
        config = extract_config(os.path.basename(file))
        if config:
            df = pd.read_excel(file)
            row_count = len(df)
            config['actual_rows'] = row_count
            row_counts[row_count].append(config)
    
    return row_counts

def print_analysis(row_counts):
    for row_count, configs in sorted(row_counts.items()):
        print(f"\nNumber of rows: {row_count}")
        print(f"Number of combinations: {len(configs)}")
        print("Configurations:")
        for config in configs:
            print(f"  - Model: {config['model_name']}")
            print(f"    Dataset: {config['dataset_name']}")
            print(f"    Typo Type: {config['typo_type']}")
            print(f"    Typo Intensity: {config['typo_intensity']}")
            print(f"    Beam Search: {config['beam_search_str']}")
            print(f"    Max New Tokens: {config['max_new_tokens']}")
            print(f"    Temperature: {config['temperature']}")
            print(f"    Strategy: {config['strategy_str']}")
            print(f"    Repeats: {config['n_repeats']}")
            print(f"    Beams: {config['n_beams']}")
            print(f"    Max Entries: {config['max_entries']}")
            print(f"    Specified Excel Rows: {config['num_excel_rows']}")
            print("    ---")

def main(exp_id):
    print(f"Analyzing experiment: {exp_id}")
    
    row_counts = analyze_excel_files(exp_id)
    print_analysis(row_counts)

if __name__ == "__main__":
    exp_id = "pert-awq-bnb-hqq-10-07"
    main(exp_id)

Analyzing experiment: pert-awq-bnb-hqq-10-07

Number of rows: 100
Number of combinations: 4800
Configurations:
  - Model: Llama-3-8B-AWQ-4bit-local
    Dataset: P27
    Typo Type: word_taxonomy_neg
    Typo Intensity: 1
    Beam Search: sample
    Max New Tokens: 25
    Temperature: 0.1
    Strategy: direct_completion
    Repeats: 1
    Beams: 5
    Max Entries: all
    Specified Excel Rows: 100
    ---
  - Model: Llama-3-8B-AWQ-4bit-local
    Dataset: P264
    Typo Type: char_substitution
    Typo Intensity: 2
    Beam Search: sample
    Max New Tokens: 25
    Temperature: 0.1
    Strategy: direct_completion
    Repeats: 1
    Beams: 5
    Max Entries: all
    Specified Excel Rows: 100
    ---
  - Model: Llama-3-8B
    Dataset: P159
    Typo Type: word_taxonomy_pos
    Typo Intensity: 1
    Beam Search: sample
    Max New Tokens: 25
    Temperature: 0.1
    Strategy: direct_completion
    Repeats: 1
    Beams: 5
    Max Entries: all
    Specified Excel Rows: 100
    ---
  - Model: Lla